# 📷 → 🧊  Reconstrucción 3D con Gaussian Splatting

Sube un **.zip con fotos** de un objeto y obtén un **modelo 3D** que puedes girar en el navegador.

Todo corre en la **GPU gratuita de Google Colab**, sin instalar nada en tu ordenador.

**Repo:** https://github.com/emr81-ua/3d-gaussian-splatting-reconstruction

---
### Cómo usarlo
1. Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.
2. Ejecuta las celdas **de arriba a abajo** (botón ▶ en cada una).
3. Cuando te lo pida, sube tu `.zip` de fotos.
4. Al final descargas el modelo `.ply` y lo ves online.

> ⏱️ La instalación tarda unos minutos y el entrenamiento otros tantos. Paciencia con la primera celda.


## 1. Comprobar la GPU


In [ ]:
# Debe aparecer una GPU (T4). Si dice 'command not found' o no hay GPU,
# ve a  Entorno de ejecucion -> Cambiar tipo de entorno -> GPU.
!nvidia-smi


## 2. Instalar las herramientas
COLMAP (poses de cámara) + nerfstudio (entrenamiento 3D Gaussian Splatting).


In [ ]:
# Instalacion (varios minutos la primera vez)
!sudo apt-get -qq update
!sudo apt-get -qq install -y colmap
!pip install -q nerfstudio
print('
Instalacion terminada.')


## 3. Sube tu .zip de fotos
30–60 fotos dando la vuelta al objeto, con solape entre una y la siguiente.


In [ ]:
import zipfile, os, glob, shutil
from google.colab import files

shutil.rmtree('/content/photos', ignore_errors=True)
shutil.rmtree('/content/_unzip', ignore_errors=True)
os.makedirs('/content/photos', exist_ok=True)

print('Sube tu .zip de fotos:')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/_unzip')

exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
n = 0
for root, _, fs in os.walk('/content/_unzip'):
    for f in sorted(fs):
        if f.lower().endswith(exts):
            ext = os.path.splitext(f)[1].lower()
            shutil.copy(os.path.join(root, f), f'/content/photos/{n:04d}{ext}')
            n += 1
print(f'
{n} fotos listas en /content/photos')


## 4. Estimar poses de cámara (COLMAP)
Detecta puntos, empareja vistas y calcula desde dónde se tomó cada foto.


In [ ]:
!ns-process-data images --data /content/photos --output-dir /content/processed


## 5. Entrenar el modelo 3D Gaussian Splatting
Puedes bajar `--max-num-iterations` a 7000 para una prueba más rápida.


In [ ]:
!ns-train splatfacto \
    --data /content/processed \
    --max-num-iterations 15000 \
    --viewer.quit-on-train-completion True \
    --output-dir /content/outputs


## 6. Exportar el modelo a .ply


In [ ]:
import glob
configs = sorted(glob.glob('/content/outputs/**/config.yml', recursive=True))
assert configs, 'No encuentro config.yml: revisa que el entrenamiento (celda 5) termino bien.'
config = configs[-1]
print('Usando config:', config)
!ns-export gaussian-splat --load-config "$config" --output-dir /content/export


## 7. Descargar el modelo


In [ ]:
import glob
from google.colab import files
plys = sorted(glob.glob('/content/export/**/*.ply', recursive=True))
assert plys, 'No se genero ningun .ply en /content/export.'
modelo = plys[-1]
print('Modelo 3D:', modelo, '(', round(os.path.getsize(modelo)/1e6, 1), 'MB )')
files.download(modelo)


## 8. Verlo online
Arrastra el `.ply` que acabas de descargar a cualquiera de estos visores en el navegador:

- **SuperSplat**: https://playcanvas.com/supersplat/editor
- **antimatter15 splat viewer**: https://antimatter15.com/splat/

---
### ⚠️ Notas
- Este Colab usa **nerfstudio (splatfacto)** para poder correr gratis en la nube. La versión de escritorio del proyecto usa **LichtFeld Studio**; el pipeline es equivalente.
- Si algo falla, cópiate el error y pásalo: los entornos de Colab cambian a menudo y puede hacer falta un pequeño ajuste.
